## inventaire_fichiers_familles_bofip

**Fichier(s) source :** `./data/bofip_stock_live_20260521.tgz` (stock du 21.05.2026)

**Fichier(s) de sortie :** DataFrame en mémoire

**Description :** Inventaire des fichiers et des familles d'objets de l'archive BOFiP : nombre de fichiers et d'objets, répartition par famille, liens entre Contenu et les autres familles (references vs. requires), et renvois orphelins.

## Étape 1 — Lecture complète de l'archive

On ouvre le .tgz une seule fois et on stocke la liste de tous les fichiers en mémoire (~30 secondes).

In [1]:
import tarfile
import os
from collections import Counter, defaultdict
import pandas as pd

ARCHIVE = r"./data/bofip_stock_live_20260521.tgz"

if not os.path.isfile(ARCHIVE):
    raise FileNotFoundError(f"Fichier introuvable : {ARCHIVE}")

print(f"Archive : {os.path.basename(ARCHIVE)}")
print(f"Taille : {os.path.getsize(ARCHIVE) / (1024*1024):.1f} Mo")
print("Lecture en cours...")

tous_fichiers = []
tous_dossiers = []

with tarfile.open(ARCHIVE, "r:gz") as tar:
    for m in tar.getmembers():
        if m.isfile():
            tous_fichiers.append(m.name)
        elif m.isdir():
            tous_dossiers.append(m.name)

print(f"\nLecture terminée.")
print(f"Fichiers : {len(tous_fichiers):,}")
print(f"Dossiers : {len(tous_dossiers):,}")

Archive : bofip_stock_live_20260521.tgz
Taille : 111.0 Mo
Lecture en cours...

Lecture terminée.
Fichiers : 14,993
Dossiers : 15,095


## Étape 2 — Noms de fichiers les plus fréquents

In [2]:
basenames = [os.path.basename(f) for f in tous_fichiers]

print("Les 10 noms de fichiers les plus fréquents :")
for nom, n in Counter(basenames).most_common(10):
    print(f"  {nom:25s} : {n:,}")

Les 10 noms de fichiers les plus fréquents :
  document.xml              : 7,495
  data.html                 : 6,311
  data1.jpg                 : 532
  data1.JPG                 : 190
  data1.png                 : 153
  data1.pdf                 : 112
  data1                     : 81
  data1.PNG                 : 63
  data1.jpeg                : 27
  data1.odt                 : 10


## Étape 3 — Structure de l'archive : comprendre le chemin

L'archive est organisée ainsi :
```
Contenu/1007-PGP/2022-03-23/document.xml
Contenu/1007-PGP/2022-03-23/data.html
Image/5102-PGP/2013-06-17/document.xml
Image/5102-PGP/2013-06-17/data1.JPG
```

Le chemin se lit : `Famille / Identifiant-PGP / Date-version / fichier`. L'identifiant de l'objet est l'**avant-dernier** niveau du chemin (le code PGP), pas le dernier (la date).

In [3]:
# Vérification : afficher 5 chemins de document.xml pour voir la structure
exemples = [f for f in tous_fichiers if f.endswith("document.xml")][:5]
print("Exemples de chemins document.xml :")
for ex in exemples:
    parts = ex.replace("\\", "/").split("/")
    print(f"  {ex}")
    print(f"    → famille: {parts[-4] if len(parts)>=4 else '?'}, identifiant: {parts[-3] if len(parts)>=3 else '?'}, date: {parts[-2] if len(parts)>=2 else '?'}")
    print()

Exemples de chemins document.xml :
  BOFiP/documents/Contenu/Commentaire/TVA/1000-PGP/2023-01-18/document.xml
    → famille: TVA, identifiant: 1000-PGP, date: 2023-01-18

  BOFiP/documents/Contenu/Commentaire/TVA/1001-PGP/2023-01-18/document.xml
    → famille: TVA, identifiant: 1001-PGP, date: 2023-01-18

  BOFiP/documents/Contenu/Commentaire/TVA/1007-PGP/2022-03-23/document.xml
    → famille: TVA, identifiant: 1007-PGP, date: 2022-03-23

  BOFiP/documents/Contenu/Commentaire/TVA/1009-PGP/2019-05-15/document.xml
    → famille: TVA, identifiant: 1009-PGP, date: 2019-05-15

  BOFiP/documents/Contenu/Commentaire/TVA/1010-PGP/2025-04-16/document.xml
    → famille: TVA, identifiant: 1010-PGP, date: 2025-04-16



## Étape 4 — Identification des objets et des familles

Un objet est un dossier qui contient un `document.xml`. L'identifiant est le **code PGP** (avant-dernier niveau).

In [4]:
# Regrouper les fichiers par dossier parent
par_dossier = defaultdict(list)
for f in tous_fichiers:
    dossier = os.path.dirname(f).replace("\\", "/")
    par_dossier[dossier].append(os.path.basename(f))

# Identifier chaque objet
objets = []
for dossier, fichiers in par_dossier.items():
    if "document.xml" not in fichiers:
        continue
    
    parts = dossier.split("/")
    
    # L'identifiant PGP est l'avant-dernier niveau
    # Ex: bofip_stock_live/Contenu/1007-PGP/2022-03-23 → identifiant = 1007-PGP
    if len(parts) >= 2:
        identifiant = parts[-2]  # avant-dernier = code PGP
        date_version = parts[-1]  # dernier = date
    else:
        identifiant = parts[-1]
        date_version = ""
    
    # Détecter la famille
    d_lower = dossier.lower()
    if "/contenu/" in d_lower or d_lower.endswith("/contenu"):
        famille = "Contenu"
    elif "/image/" in d_lower or d_lower.endswith("/image"):
        famille = "Image"
    elif "/attachment/" in d_lower or "/fichier/" in d_lower:
        famille = "Attachment"
    elif "/planclassement/" in d_lower or "/plan/" in d_lower:
        famille = "PlanClassement"
    else:
        # Fallback : essayer le niveau au-dessus de l'identifiant
        if len(parts) >= 3:
            fam_candidate = parts[-3].lower()
            if "contenu" in fam_candidate:
                famille = "Contenu"
            elif "image" in fam_candidate:
                famille = "Image"
            elif "attachment" in fam_candidate or "fichier" in fam_candidate:
                famille = "Attachment"
            elif "plan" in fam_candidate:
                famille = "PlanClassement"
            else:
                famille = "Autre"
        else:
            famille = "Autre"
    
    objets.append({
        "identifiant": identifiant,
        "famille": famille,
        "date_version": date_version,
        "chemin": dossier,
        "nb_fichiers": len(fichiers),
        "a_html": "data.html" in fichiers,
        "a_pdf": any(f.endswith(".pdf") for f in fichiers),
        "a_jpg": any(f.upper().endswith(".JPG") or f.upper().endswith(".JPEG") for f in fichiers),
        "liste_fichiers": ", ".join(sorted(fichiers))
    })

df = pd.DataFrame(objets)
print(f"Total objets : {len(df):,}")
print()
for fam, n in df["famille"].value_counts().items():
    print(f"  {fam:20s} : {n:,} objets")

# Vérification : les identifiants sont-ils des codes PGP ?
print()
pgp_count = df["identifiant"].str.contains("-PGP", case=False).sum()
print(f"Identifiants contenant '-PGP' : {pgp_count:,} / {len(df):,}")
print()
print("5 premiers identifiants Contenu :")
for _, r in df[df["famille"]=="Contenu"].head(5).iterrows():
    print(f"  {r['identifiant']:15s} (date: {r['date_version']}, fichiers: {r['liste_fichiers']})")

Total objets : 7,495

  Contenu              : 6,311 objets
  Image                : 1,041 objets
  Attachment           : 142 objets
  PlanClassement       : 1 objets

Identifiants contenant '-PGP' : 7,495 / 7,495

5 premiers identifiants Contenu :
  1000-PGP        (date: 2023-01-18, fichiers: data.html, document.xml)
  1001-PGP        (date: 2023-01-18, fichiers: data.html, document.xml)
  1007-PGP        (date: 2022-03-23, fichiers: data.html, document.xml)
  1009-PGP        (date: 2019-05-15, fichiers: data.html, document.xml)
  1010-PGP        (date: 2025-04-16, fichiers: data.html, document.xml)


## Étape 5 — Tableau récapitulatif : objets vs. fichiers

In [5]:
recap = df.groupby("famille").agg(
    objets=("identifiant", "count"),
    fichiers=("nb_fichiers", "sum"),
    avec_html=("a_html", "sum"),
    avec_jpg=("a_jpg", "sum"),
    avec_pdf=("a_pdf", "sum")
).reset_index()
recap = recap.sort_values("objets", ascending=False)
recap["ratio"] = (recap["fichiers"] / recap["objets"]).round(1)

total_row = pd.DataFrame([{
    "famille": "TOTAL",
    "objets": recap["objets"].sum(),
    "fichiers": recap["fichiers"].sum(),
    "avec_html": int(recap["avec_html"].sum()),
    "avec_jpg": int(recap["avec_jpg"].sum()),
    "avec_pdf": int(recap["avec_pdf"].sum()),
    "ratio": ""
}])
recap_affich = pd.concat([recap, total_row], ignore_index=True)

print("=" * 80)
print("TABLEAU — Objets et fichiers par famille")
print("=" * 80)
print(recap_affich.to_string(index=False))
print("=" * 80)

TABLEAU — Objets et fichiers par famille
       famille  objets  fichiers  avec_html  avec_jpg  avec_pdf ratio
       Contenu    6311     12622       6311         0         0   2.0
         Image    1041      2082          0       749         0   2.0
    Attachment     142       284          0         0       112   2.0
PlanClassement       1         2          0         0         0   2.0
         TOTAL    7495     14990       6311       749       112      


## Étape 6 — Exemples concrets par famille

In [6]:
for fam in ["Contenu", "Image", "Attachment", "PlanClassement"]:
    sous = df[df["famille"] == fam].head(3)
    if len(sous) > 0:
        print(f"\n--- {fam} (3 premiers) ---")
        for _, r in sous.iterrows():
            print(f"  {r['identifiant']:15s} | date: {r['date_version']} | {r['liste_fichiers']}")


--- Contenu (3 premiers) ---
  1000-PGP        | date: 2023-01-18 | data.html, document.xml
  1001-PGP        | date: 2023-01-18 | data.html, document.xml
  1007-PGP        | date: 2022-03-23 | data.html, document.xml

--- Image (3 premiers) ---
  10038-PGP       | date: 2020-04-15 | data1.JPG, document.xml
  10040-PGP       | date: 2016-03-02 | data1.JPG, document.xml
  10041-PGP       | date: 2016-03-02 | data1.JPG, document.xml

--- Attachment (3 premiers) ---
  10099-PGP       | date: 2015-05-06 | data1.pdf, document.xml
  10158-PGP       | date: 2015-07-01 | data1.pdf, document.xml
  10223-PGP       | date: 2015-08-05 | data1.pdf, document.xml

--- PlanClassement (3 premiers) ---
  8-PGP           | date: 2026-05-21 | data.xml, document.xml


## Étape 7 — Liens entre Contenu et les autres familles

Deuxième passe sur l'archive : on lit les `document.xml` des Contenu pour extraire les `dc:relation`. Cette cellule prend 1 à 2 minutes.

In [7]:
import xml.etree.ElementTree as ET

# Construire l'ensemble des chemins document.xml des Contenu
chemins_contenu = set()
for _, r in df[df["famille"] == "Contenu"].iterrows():
    chemins_contenu.add(r["chemin"] + "/document.xml")

print(f"Document.xml Contenu à lire : {len(chemins_contenu):,}")
print("Lecture en cours (1-2 minutes)...")

liens = []
erreurs = 0
compteur = 0

with tarfile.open(ARCHIVE, "r:gz") as tar:
    for m in tar.getmembers():
        if not m.isfile():
            continue
        chemin = m.name.replace("\\", "/")
        if chemin not in chemins_contenu:
            continue
        
        # Identifiant PGP = avant-dernier niveau
        parts = chemin.split("/")
        source_id = parts[-3] if len(parts) >= 3 else "inconnu"
        
        try:
            f = tar.extractfile(m)
            if f is None:
                continue
            tree = ET.parse(f)
            root = tree.getroot()
            
            for elem in root.iter():
                tag = elem.tag.split("}")[-1] if "}" in elem.tag else elem.tag
                if tag == "relation" and elem.text:
                    texte = elem.text.strip()
                    rel_type = ""
                    for k, v in elem.attrib.items():
                        if "type" in k.lower():
                            rel_type = v
                            break
                    
                    if ":" in texte:
                        fam_cible, id_cible = texte.split(":", 1)
                    else:
                        fam_cible, id_cible = "Inconnu", texte
                    
                    liens.append({
                        "source": source_id,
                        "type_relation": rel_type,
                        "cible_famille": fam_cible,
                        "cible_id": id_cible
                    })
            compteur += 1
            if compteur % 1000 == 0:
                print(f"  {compteur:,} documents lus...")
        except Exception:
            erreurs += 1

df_liens = pd.DataFrame(liens)
print(f"\nTerminé. {compteur:,} documents lus, {len(df_liens):,} liens extraits.")
if erreurs > 0:
    print(f"Erreurs : {erreurs}")

Document.xml Contenu à lire : 6,311
Lecture en cours (1-2 minutes)...
  1,000 documents lus...
  2,000 documents lus...
  3,000 documents lus...
  4,000 documents lus...
  5,000 documents lus...
  6,000 documents lus...

Terminé. 6,311 documents lus, 23,128 liens extraits.


## Étape 8 — Tableau croisé : references vs. requires

In [8]:
if len(df_liens) > 0:
    print("Liens par famille cible :")
    print(df_liens["cible_famille"].value_counts().to_string())
    print()
    print("Liens par type de relation :")
    print(df_liens["type_relation"].value_counts().to_string())
    print()
    print("Tableau croisé : famille cible × type de relation")
    print(pd.crosstab(df_liens["cible_famille"], df_liens["type_relation"], margins=True).to_string())
else:
    print("Aucun lien extrait.")

Liens par famille cible :
cible_famille
Contenu      21403
Actualite     1072
Image          536
Fichier        117

Liens par type de relation :
type_relation
references    22475
requires        653

Tableau croisé : famille cible × type de relation
type_relation  references  requires    All
cible_famille                             
Actualite            1072         0   1072
Contenu             21403         0  21403
Fichier                 0       117    117
Image                   0       536    536
All                 22475       653  23128


## Étape 9 — Orphelins : quels liens ne mènent nulle part ?

In [9]:
if len(df_liens) > 0:
    # Ensemble de tous les identifiants PGP présents dans l'archive
    ids_presents = set(df["identifiant"].values)
    
    # Vérification : les cibles contiennent-elles des codes PGP ?
    print("Vérification des formats :")
    print(f"  Identifiants dans l'archive (5 ex.) : {list(ids_presents)[:5]}")
    print(f"  Cibles des liens (5 ex.) : {df_liens['cible_id'].head(5).tolist()}")
    print()
    
    # Vérifier la présence de chaque cible
    df_liens["cible_presente"] = df_liens["cible_id"].isin(ids_presents)
    
    resolus = df_liens[df_liens["cible_presente"]]
    orphelins = df_liens[~df_liens["cible_presente"]]
    
    print(f"Liens résolus (cible présente dans l'archive) : {len(resolus):,}")
    print(f"Liens orphelins (cible absente) : {len(orphelins):,} ({len(orphelins)/len(df_liens)*100:.1f} %)")
    print()
    print("Orphelins par famille cible :")
    print(orphelins["cible_famille"].value_counts().to_string())
else:
    print("Pas de liens à vérifier.")

Vérification des formats :
  Identifiants dans l'archive (5 ex.) : ['7088-PGP', '7528-PGP', '5202-PGP', '5314-PGP', '5705-PGP']
  Cibles des liens (5 ex.) : ['2412-PGP', '2418-PGP', '2421-PGP', '5992-PGP', '1000-PGP']

Liens résolus (cible présente dans l'archive) : 21,774
Liens orphelins (cible absente) : 1,354 (5.9 %)

Orphelins par famille cible :
cible_famille
Actualite    1071
Contenu       265
Fichier        15
Image           3


## Étape 10 — Documents Contenu reliés à d'autres familles

In [10]:
if len(df_liens) > 0:
    print("Combien de documents Contenu renvoient vers chaque famille :")
    for fam in sorted(df_liens["cible_famille"].unique()):
        sous = df_liens[df_liens["cible_famille"] == fam]
        n_docs = sous["source"].nunique()
        n_liens = len(sous)
        n_orphelins = len(sous[~sous["cible_presente"]])
        print(f"  vers {fam:15s} : {n_docs:>5,} documents, {n_liens:>6,} liens, dont {n_orphelins:>5,} orphelins")

Combien de documents Contenu renvoient vers chaque famille :
  vers Actualite       : 1,066 documents,  1,072 liens, dont 1,071 orphelins
  vers Contenu         : 4,879 documents, 21,403 liens, dont   265 orphelins
  vers Fichier         :    54 documents,    117 liens, dont    15 orphelins
  vers Image           :   198 documents,    536 liens, dont     3 orphelins


## Étape 11 — Synthèse

In [11]:
print("=" * 70)
print("SYNTHÈSE")
print("=" * 70)
print(f"Archive : {os.path.basename(ARCHIVE)}")
print(f"Fichiers dans l'archive : {len(tous_fichiers):,}")
print(f"Objets (avec document.xml) : {len(df):,}")
print(f"Ratio : {len(tous_fichiers)/len(df):.1f} fichiers par objet")
print()
print("Répartition par famille :")
for fam in ["Contenu", "Image", "Attachment", "PlanClassement"]:
    n = len(df[df["famille"] == fam])
    print(f"  {fam:20s} : {n:>6,} objets")
print(f"  {'TOTAL':20s} : {len(df):>6,} objets")
print()
if len(df_liens) > 0:
    print(f"Liens déclarés par les Contenu : {len(df_liens):,}")
    print(f"  references (citations) : {len(df_liens[df_liens['type_relation']=='references']):,}")
    print(f"  requires (dépendances) : {len(df_liens[df_liens['type_relation']=='requires']):,}")
    print(f"Liens orphelins : {len(orphelins):,} ({len(orphelins)/len(df_liens)*100:.1f} %)")
    print()
    print("Orphelins par famille cible :")
    for fam, n in orphelins["cible_famille"].value_counts().items():
        print(f"  {fam:15s} : {n:,}")
print()
print("Conclusion :")
print(f"  {len(tous_fichiers):,} fichiers = {len(df):,} objets × 2 fichiers/objet")
print(f"  Seuls les {len(df[df['famille']=='Contenu']):,} Contenu portent de la doctrine.")
print(f"  Les {len(df[df['famille']=='Image']):,} Images et {len(df[df['famille']=='Attachment']):,} Attachments sont des ressources liées (requires).")
print("=" * 70)

SYNTHÈSE
Archive : bofip_stock_live_20260521.tgz
Fichiers dans l'archive : 14,993
Objets (avec document.xml) : 7,495
Ratio : 2.0 fichiers par objet

Répartition par famille :
  Contenu              :  6,311 objets
  Image                :  1,041 objets
  Attachment           :    142 objets
  PlanClassement       :      1 objets
  TOTAL                :  7,495 objets

Liens déclarés par les Contenu : 23,128
  references (citations) : 22,475
  requires (dépendances) : 653
Liens orphelins : 1,354 (5.9 %)

Orphelins par famille cible :
  Actualite       : 1,071
  Contenu         : 265
  Fichier         : 15
  Image           : 3

Conclusion :
  14,993 fichiers = 7,495 objets × 2 fichiers/objet
  Seuls les 6,311 Contenu portent de la doctrine.
  Les 1,041 Images et 142 Attachments sont des ressources liées (requires).
